# 05 · IC timeseries from GEE (Moscardo catchment)

This notebook demonstrates `GEEFetcher.fetch_timeseries()` and computes a temporal sequence of Connectivity Index (IC) maps for the Moscardo catchment (Italian Alps).

## 1 · Configuration

In [3]:
from pathlib import Path

# Small area in Chambal Catchment, India
BOUNDS = (76.09556906553738, 24.867882347223897, 76.52756888567045, 25.178609604000833) #<-- Chambal

# Use projected CRS in meters for slope/routing reliability
CRS = "EPSG:32643"

START_DATE = "1990-01-01"
END_DATE = "2025-12-31"

# Temporal resampling options: 'monthly' | 'seasonal' | 'annual'
TEMPORAL_RESAMPLING = "annual"

# DEM/NDVI target fetch scale in meters
SCALE = 90

FLOW_DIRECTOR = "DINF"
W_MIN = 0.005
W_MAX = 1.0

# Replace if your EE setup requires an explicit project
GEE_PROJECT = "drylands-aberuni"

OUT_DIR = Path(f"output_nb5_scale{SCALE}m")
OUT_DIR.mkdir(exist_ok=True)
OUT_DIR

PosixPath('output_nb5_scale90m')

In [4]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import rioxarray

from landlab import RasterModelGrid
from geomorphconn.gee import GEEFetcher
from geomorphconn.components import ConnectivityIndex
from geomorphconn.weights import preset_rainfall_ndvi
from geomorphconn.analysis import classify_dynamic_crus
from geomorphconn.analysis.cru_dynamics import CRU_CLASSES
from geomorphconn.analysis.utils import export_cru_geotiff, plot_cru_map

In [5]:
GEEFetcher.list_sources()

DEM sources:         ['SRTM', 'COPDEM30', 'MERIT']
Rainfall sources:    ['CHIRPS', 'ERA5', 'PERSIANN']
NDVI sources:        ['SENTINEL2', 'LANDSAT8', 'LANDSAT9', 'LANDSAT7', 'LANDSAT5', 'LANDSATALL']
Land-cover sources:  ['WORLDCOVER', 'WORLDCOVER_V100', 'MODIS_LC']


## 2 · Fetch NDVI/Rainfall timeseries windows from GEE

In [6]:
fetcher = GEEFetcher(
    bounds=BOUNDS,
    dem_source="COPDEM30",
    rainfall_source="CHIRPS",
    ndvi_source="LANDSATALL",
    start_date=START_DATE,
    end_date=END_DATE,
    scale=SCALE,
    crs=CRS,
    gee_project=GEE_PROJECT,
)

ts = fetcher.fetch_timeseries(resampling=TEMPORAL_RESAMPLING)
dem = ts["dem"]
profile = ts["profile"]
periods = ts["periods"]

nrows, ncols = dem.shape
print(f"Fetched {len(periods)} periods ({TEMPORAL_RESAMPLING})")
print(f"DEM shape: {dem.shape}, dx={abs(profile['transform'].a):.2f} m")
print("First period:", periods[0]["label"], periods[0]["start_date"], periods[0]["end_date"])

GEEFetcher configured:
  DEM        : Copernicus DEM GLO-30 (~30 m)
  Rainfall   : CHIRPS daily precipitation (~5.5 km)
  NDVI       : All Landsat C2 SR merged (L5 TM + L7 ETM+ SLC-on + L8 OLI + L9 OLI-2), 30 m, 1984–present
  Land cover : None (not fetched)
  Period     : 1990-01-01 → 2025-12-31
  Scale      : 90 m  |  CRS: EPSG:32643
Fetching DEM (shared across all periods) …
  DEM shape: 387×481, range: 215.3–325.8 m
[1/36] Fetching window 1990: 1990-01-01 → 1990-12-31
  NDVI shape: (387, 481), range: -0.089–0.395 (LANDSATALL merged)
  Rainfall shape: (387, 481), range: 815.68–985.48
[2/36] Fetching window 1991: 1991-01-01 → 1991-12-31
  NDVI shape: (387, 481), range: -0.045–0.368 (LANDSATALL merged)
  Rainfall shape: (387, 481), range: 513.92–635.97
[3/36] Fetching window 1992: 1992-01-01 → 1992-12-31
  NDVI shape: (387, 481), range: -0.061–0.363 (LANDSATALL merged)
  Rainfall shape: (387, 481), range: 517.25–608.29
[4/36] Fetching window 1993: 1993-01-01 → 1993-12-31
  NDVI shape:

ValueError: max() arg is an empty sequence

## 3 · Build Landlab grid once (shared DEM)

In [ ]:
tr = profile["transform"]
x_min = tr.c
y_min = tr.f - nrows * abs(tr.e)
dx = abs(tr.a)

grid = RasterModelGrid(
    (nrows, ncols),
    xy_spacing=dx,
    xy_of_lower_left=(x_min, y_min),
)

dem_ll = np.flipud(dem).copy()
dem_ll[np.isnan(dem_ll)] = np.nanmin(dem_ll) - 1.0
grid.add_field("topographic__elevation", dem_ll.ravel(), at="node")

print(f"Grid nodes: {grid.number_of_nodes:,}")

## 4 · Compute IC for each period

For each temporal window, IC is computed with rainfall+NDVI weight (`preset_rainfall_ndvi`).

In [ ]:
def to_node(arr2d):
    a = np.flipud(arr2d).copy()
    a[np.isnan(a)] = 0.0
    return a.ravel()

records = []
ic_maps = []

for p in periods:
    ndvi_nodes = to_node(p["ndvi"])

    rf_nodes = to_node(p["rainfall"])

    wb = preset_rainfall_ndvi(rf_nodes, ndvi_nodes, w_min=W_MIN)
    ic = ConnectivityIndex(
        grid,
        flow_director=FLOW_DIRECTOR,
        weight=wb,
        w_min=W_MIN,
        w_max=W_MAX,
    )
    ic.run_one_step()

    # Copy the field snapshot so later timesteps do not overwrite earlier maps.
    ic_arr = ic.IC.reshape(grid.shape).copy()
    ic_maps.append(ic_arr)

    vals = ic_arr[np.isfinite(ic_arr)]
    records.append({
        "label": p["label"],
        "start_date": p["start_date"],
        "end_date": p["end_date"],
        "ic_mean": float(np.mean(vals)),
        "ic_median": float(np.median(vals)),
        "ic_p10": float(np.percentile(vals, 10)),
        "ic_p90": float(np.percentile(vals, 90)),
        "ic_std": float(np.std(vals)),
    })

df_ts = pd.DataFrame(records)
df_ts

## 5 · Temporal IC behaviour plots

In [ ]:
x = np.arange(len(df_ts))

fig, ax = plt.subplots(figsize=(11, 5), dpi=120)
ax.plot(x, df_ts["ic_mean"], marker="o", label="IC mean", linewidth=2)
ax.plot(x, df_ts["ic_median"], marker="s", label="IC median", linewidth=2)
ax.fill_between(x, df_ts["ic_p10"], df_ts["ic_p90"], alpha=0.2, label="P10-P90")

ax.set_xticks(x)
ax.set_xticklabels(df_ts["label"], rotation=45, ha="right")
ax.set_ylabel("IC")
ax.set_title(f"Temporal IC summary ({TEMPORAL_RESAMPLING})")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Distribution-by-period boxplot
flat = []
for i, label in enumerate(df_ts["label"]):
    vals = ic_maps[i][np.isfinite(ic_maps[i])]
    sample = vals if vals.size <= 25000 else vals[np.random.choice(vals.size, 25000, replace=False)]
    flat.extend([(label, v) for v in sample])

df_box = pd.DataFrame(flat, columns=["label", "IC"])

fig, ax = plt.subplots(figsize=(12, 5), dpi=120)
df_box.boxplot(column="IC", by="label", ax=ax, rot=45, grid=False)
ax.set_title("IC distribution by period")
ax.set_xlabel("Period")
ax.set_ylabel("IC")
plt.suptitle("")
plt.tight_layout(); plt.show()

## 6 · Map snapshots for selected periods

In [ ]:
n_show = min(4, len(ic_maps))
idxs = np.linspace(0, len(ic_maps) - 1, n_show).round().astype(int)

fig, axes = plt.subplots(1, n_show, figsize=(4*n_show, 4), dpi=120)
if n_show == 1:
    axes = [axes]

vmin = np.nanpercentile(np.stack(ic_maps), 5)
vmax = np.nanpercentile(np.stack(ic_maps), 95)
im = None

for ax, i in zip(axes, idxs):
    im = ax.imshow(np.flipud(ic_maps[i]), cmap="RdYlGn", vmin=vmin, vmax=vmax, interpolation="nearest")
    ax.set_title(df_ts.loc[i, "label"])
    ax.axis("off")

if im is not None:
    fig.colorbar(im, ax=axes, shrink=0.6, label="IC", pad=-0.8)
plt.suptitle("IC map evolution through time", fontweight="bold")
# plt.tight_layout(); plt.show()

## 7 · Dynamic CRU classification from the IC time series

This section classifies temporal IC behaviour into dynamic Connectivity Response Units (CRUs) using `geomorphconn.analysis.classify_dynamic_crus`.

In [ ]:
time_coord = pd.to_datetime(df_ts["start_date"])

ic_cube = xr.DataArray(
    np.stack([np.flipud(arr) for arr in ic_maps]),
    coords={"time": time_coord, "y": np.arange(nrows), "x": np.arange(ncols)},
    dims=("time", "y", "x"),
    name="IC",
)
ic_cube = ic_cube.rio.write_crs(profile["crs"]).rio.write_transform(profile["transform"])

recent_window = 1 # if len(ic_maps) >= 4 else 1
cru_map = classify_dynamic_crus(
    ic_cube,
    recent_window=recent_window,
    early_window=recent_window,
    attribution_tags={"source": "GeomorphConn nb5 demo"},
)

cru_counts = (
    pd.Series(cru_map.values.ravel(), name="cru_code")
    .dropna()
    .astype(int)
    .value_counts()
    .sort_index()
    .rename_axis("cru_code")
    .reset_index(name="n_pixels")
)
cru_counts["class_name"] = cru_counts["cru_code"].map(CRU_CLASSES)
display(cru_counts)

fig = plot_cru_map(
    cru_map,
    title=f"Dynamic CRU classification ({TEMPORAL_RESAMPLING})",
    figsize=(11, 6),
    cbar_label="CRU class",
)
plt.tight_layout()
plt.show()

cru_out = OUT_DIR / f"cru_dynamic_{TEMPORAL_RESAMPLING}.tif"
cru_paths = export_cru_geotiff(cru_map, cru_out)
cru_counts_out = OUT_DIR / f"cru_dynamic_{TEMPORAL_RESAMPLING}_summary.csv"
cru_counts.to_csv(cru_counts_out, index=False)

print("Saved CRU raster:", cru_paths["tif"])
print("Saved CRU summary:", cru_counts_out.resolve())

In [ ]:
np.unique(cru_map, return_counts=True)

In [20]:
df_ts.to_csv(OUT_DIR / f"ic_timeseries_{TEMPORAL_RESAMPLING}.csv", index=False)
print("Saved:", (OUT_DIR / f"ic_timeseries_{TEMPORAL_RESAMPLING}.csv").resolve())

Saved: /mnt/e/SideResearch/softwares/IndexOfConnectivity/sedconn_v3/sedconn/notebooks/output_nb5/ic_timeseries_monthly.csv


In [24]:
fetcher.list_sources()

DEM sources:         ['SRTM', 'COPDEM30', 'MERIT']
Rainfall sources:    ['CHIRPS', 'ERA5', 'PERSIANN']
NDVI sources:        ['SENTINEL2', 'LANDSAT8', 'LANDSAT9']
Land-cover sources:  ['WORLDCOVER', 'WORLDCOVER_V100', 'MODIS_LC']
